# Batch processing I22 SAXS/WAXS data using the MoDaCor server

This notebook processes a batch of I22 SAXS/WAXS measurements with the MoDaCor runtime server. It extends the simpler `run_pipeline_job` notebook: use that notebook for a first single-file run, and use this server workflow when many similar files should reuse cached runtime state.

The WAXS pipeline and the SAXS pipeline are instantiated in separate runtime sessions on the server, and configured 
with their respective pipelines. 

This example is currently developed against MoDaCor 1.7.0. The repository README records the development baseline; the first archival release will pin an immutable MoDaCor revision after the complete SAXS/WAXS workflow is rerun. The included DAWN cross-check pipelines deliberately reproduce selected DAWN behaviour, while the `solids_operando` pipelines represent the recommended physical corrections.

Run the cells from top to bottom. For normal use, only edit **User Configuration**.

Fresh environment setup from a terminal:

```bash
cd /path/to/MoDaCor_examples
uv venv --python 3.14 .venv
source .venv/bin/activate
uv pip install -e "/path/to/MoDaCor[server,attenuation,plotting,tiled-tests]" requests matplotlib ipykernel hdf5plugin
python -m ipykernel install --user --name modacor-i22 --display-name "Python (MoDaCor I22)"
```

After installing, select the `Python (MoDaCor I22)` kernel for this notebook.


## Optional Notebook Install Cell

Run the next cell only if this notebook kernel cannot import MoDaCor, FastAPI, uvicorn, requests, matplotlib, or the optional Tiled dependencies used by Example 4.


In [ ]:
# Use the terminal setup above when imports fail, then restart the notebook kernel.
# Keeping environment creation outside the notebook makes the selected MoDaCor
# revision explicit and reproducible.


## User Configuration

Start Jupyter from the examples repository root or this instrument directory. The packaged data and pipelines are discovered automatically. The sample discovery is tolerant: if files are still copying, the notebook will warn and still let you preview the pipeline graph and start the server.


In [ ]:
from pathlib import Path

def locate_example_dir(relative_path):
    start = Path.cwd().resolve()
    for parent in (start, *start.parents):
        for candidate in (parent, parent / relative_path):
            if (candidate / "data-manifest.json").is_file() and (candidate / "pipelines").is_dir():
                return candidate
    raise FileNotFoundError(
        "Could not locate DLS/I22. Start Jupyter from the examples repository root "
        "or from the DLS/I22 instrument directory."
    )


PROJECT_DIR = locate_example_dir(Path("DLS") / "I22")
PIPELINE_PATHS = {
    "SAXS": PROJECT_DIR / "pipelines" / "I22_SAXS_solids_operando.yaml",
    "WAXS": PROJECT_DIR / "pipelines" / "I22_WAXS_solids_operando.yaml",
    # "SAXS": PROJECT_DIR / "pipelines" / "I22_SAXS_DAWN_crosscheck.yaml",
    # "WAXS": PROJECT_DIR / "pipelines" / "I22_WAXS_DAWN_crosscheck.yaml",
}

DATA_ROOT = PROJECT_DIR / "data"
EXAMPLE_PROCESSING_DIR = DATA_ROOT / "processing"
CALIBRATION_FILES = {
    "SAXS": EXAMPLE_PROCESSING_DIR / "SAXS_calibration.nxs",
    "WAXS": EXAMPLE_PROCESSING_DIR / "WAXS_calibration.nxs",
}
MASK_FILES = {
    "SAXS": EXAMPLE_PROCESSING_DIR / "SAXS_mask.nxs",
    "WAXS": EXAMPLE_PROCESSING_DIR / "WAXS_mask.nxs",
}

# Select which detector sessions to create and run.
DETECTORS_TO_RUN = ["SAXS", "WAXS"]
DETECTOR = DETECTORS_TO_RUN[0]  # selected detector for single-detector previews
BACKGROUND_FILES = {
    "SAXS": DATA_ROOT / "i22-977723.nxs",
    "WAXS": DATA_ROOT / "i22-977723.nxs",
}

WORK_DIR = PROJECT_DIR / "work"
PREPROCESSED_DATA_DIR = WORK_DIR / "preprocessed"
PREPROCESSED_CALIBRATION_DIR = WORK_DIR / "preprocessed_calibration"
OUTPUT_DIR = WORK_DIR / "output"

PIPELINE_PATH = PIPELINE_PATHS[DETECTOR]
BACKGROUND_FILE = BACKGROUND_FILES[DETECTOR]
SESSION_IDS = {detector: f"i22-{detector.lower()}-server-batch" for detector in DETECTORS_TO_RUN}
SESSION_ID = SESSION_IDS[DETECTOR]
SERVER_HOST = "127.0.0.1"
SEPARATE_SERVER_PER_DETECTOR = False
SERVER_PORTS = {"SAXS": 8901, "WAXS": 8902}
SERVER_PORT = SERVER_PORTS[DETECTOR]
SERVER_LOG_PATHS = {
    detector: OUTPUT_DIR / f"modacor_server_{detector.lower()}.log"
    for detector in DETECTORS_TO_RUN
}
SERVER_LOG_PATH = SERVER_LOG_PATHS[DETECTOR]

SAMPLE_GLOB = "i22-978???.nxs"
BSDIODES_CHANNEL = 1
ABSOLUTE_INTENSITY_FACTOR = 3.8e-15
OVERWRITE_PREPROCESSED = False
MAX_SAMPLES_TO_PROCESS = None

# Set this to False while debugging so the batch stops at the first failed file.
CONTINUE_ON_SAMPLE_ERROR = False
# Use this with SEPARATE_SERVER_PER_DETECTOR to test real process-level concurrency.
PARALLEL_DETECTOR_RUNS = SEPARATE_SERVER_PER_DETECTOR
ROLLBACK_SNAPSHOT = False
RESET_SESSION_AFTER_FAILURE = True

TRACE_ENABLED = True
TRACE_WATCH = {"sample": ["signal"], "background": ["signal"]}

PLOT_SINK_REF = "plots"
RESULT_HDF_SINK_REF = "result_hdf"
LIVE_PLOT_IDS = {
    "SAXS": {"1d": "saxs-1d", "2d": "saxs-2d"},
    "WAXS": {"1d": "waxs-1d", "2d": "waxs-2d"},
}

SAMPLE_OUTPUT_DATA_PATHS = [
    "/sample/signal",
    "/sample/Q",
    # "/sample/Psi",
    # "/sample/Omega",
    # "/sample/pixel_index",
    # "/sample/mask",
]


## Import Checks And File Discovery

This cell checks imports, creates the output directory, and finds sample files. If no sample files are found yet, rerun this cell after copying finishes.


In [ ]:
import atexit
import json
import os
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import h5py
import numpy as np
import requests
from IPython.display import JSON, Markdown, display

import modacor

try:
    import hdf5plugin  # noqa: F401
except ImportError:
    print("hdf5plugin is not installed; I22 detector data compressed with Blosc will not load.")
    print("Rerun the install cell after updating MoDaCor, then restart the kernel/server.")

unknown_detectors = set(DETECTORS_TO_RUN) - {"SAXS", "WAXS"}
if unknown_detectors:
    raise ValueError(f"Unknown detector(s): {sorted(unknown_detectors)}")
if DETECTOR not in DETECTORS_TO_RUN:
    raise ValueError("DETECTOR must be included in DETECTORS_TO_RUN.")

print(f"Python: {sys.executable}")
print(f"MoDaCor: {modacor.__version__}")
print(f"MoDaCor module: {modacor.__file__}")

paths_to_check = {
    "PROJECT_DIR": PROJECT_DIR,
    "PIPELINE_PATH_SAXS": PIPELINE_PATHS["SAXS"],
    "PIPELINE_PATH_WAXS": PIPELINE_PATHS["WAXS"],
    "DATA_ROOT": DATA_ROOT,
    "SAXS_CALIBRATION": CALIBRATION_FILES["SAXS"],
    "SAXS_MASK": MASK_FILES["SAXS"],
    "WAXS_CALIBRATION": CALIBRATION_FILES["WAXS"],
    "WAXS_MASK": MASK_FILES["WAXS"],
    "BACKGROUND_FILE": BACKGROUND_FILE,
}
for path_name, path in paths_to_check.items():
    print(f"{path_name}: {path} ({'ok' if path.exists() else 'missing'})")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_CALIBRATION_DIR.mkdir(parents=True, exist_ok=True)


def is_sample_candidate(path: Path) -> bool:
    """Return True only for I22 masters with the linked inputs used here."""
    if not path.is_file() or path.resolve() == BACKGROUND_FILE.resolve():
        return False
    stem = path.stem
    required = [
        DATA_ROOT / f"{stem}-Pilatus2M_SAXS.h5",
        DATA_ROOT / f"{stem}-Pilatus2M_WAXS.h5",
        DATA_ROOT / f"{stem}-bsdiodes.h5",
    ]
    return all(candidate.is_file() for candidate in required)


sample_files = sorted(path for path in DATA_ROOT.glob(SAMPLE_GLOB) if is_sample_candidate(path))

print(f"Found {len(sample_files)} sample file(s).")
for index, sample_file in enumerate(sample_files[:10], start=1):
    print(f"{index:02d}: {sample_file}")
if len(sample_files) > 10:
    print(f"... and {len(sample_files) - 10} more")
if not sample_files:
    print("No sample files found yet. Calibration preprocessing and graph preview can still run.")


## Preprocess I22 Inputs For MoDaCor

This cell leaves the raw NeXus/HDF5 files untouched. For each measurement it writes a compact file that externally links `/entry1` from the original master, adds broadcast-ready `(images, frames, 1, 1)` normalization arrays under `/modacor/normalization`, and stores scalar calibration values under `/modacor/calibration`.

The beamstop-diode channel is reduced over its 2,000-sample axis to one mean, standard deviation, SEM, and valid-sample count per `(image, frame)`. Detector count times and transmission are reshaped or broadcast to the same detector-divisor layout.

Detector geometry is not precomputed here. The YAML pipelines point directly at the NeXus calibration files; MoDaCor resolves the detector transformation chains in `PixelCoordinates3D` and `XSGeometryFromPixelCoordinates`.


In [ ]:
PREPROCESSING_VERSION = "2026-09-02-i22-normalization-v3"


def _decode(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    if isinstance(value, np.ndarray) and value.shape == ():
        return _decode(value.item())
    return value


def _as_detector_divisor(array):
    array = np.asarray(array)
    return array.reshape(array.shape + (1, 1))


def _measurement_output_path(master_file):
    return PREPROCESSED_DATA_DIR / f"{Path(master_file).stem}_modacor.nxs"


def _needs_rewrite(output_file, *, overwrite):
    output_file = Path(output_file)
    if overwrite or not output_file.exists():
        return True
    try:
        with h5py.File(output_file, "r") as h5:
            return _decode(h5.attrs.get("preprocessing_version", "")) != PREPROCESSING_VERSION
    except OSError:
        return True


def _frame_array(values, leading_shape, *, name):
    values = np.asarray(values, dtype=float)
    if values.shape == leading_shape:
        return values
    if values.size == 1:
        return np.full(leading_shape, float(values.reshape(-1)[0]), dtype=float)
    raise ValueError(f"{name} shape {values.shape} cannot be broadcast to frame shape {leading_shape}.")


def _write_dataset(group, name, values, *, units=None, **attrs):
    dataset = group.create_dataset(name, data=values)
    if units is not None:
        dataset.attrs["units"] = units
    for key, value in attrs.items():
        dataset.attrs[key] = value
    return dataset


def _detector_shapes(source):
    return {
        "SAXS": source["/entry1/detector/data"].shape,
        "WAXS": source["/entry1/Pilatus2M_WAXS/data"].shape,
    }


def preprocess_i22_measurement(master_file, *, overwrite=False):
    """Create one compact MoDaCor-facing I22 measurement file without copying detector images."""
    master_file = Path(master_file).resolve()
    output_file = _measurement_output_path(master_file)
    if not _needs_rewrite(output_file, overwrite=overwrite):
        return output_file

    with h5py.File(master_file, "r") as source:
        bsdiodes = np.asarray(source["/entry1/bsdiodes/data"][()], dtype=float)
        if bsdiodes.ndim != 4 or BSDIODES_CHANNEL >= bsdiodes.shape[-1]:
            raise ValueError(f"Unexpected bsdiodes shape {bsdiodes.shape}; channel {BSDIODES_CHANNEL} is unavailable.")

        detector_shapes = _detector_shapes(source)
        leading_shape = tuple(detector_shapes["SAXS"][:-2])
        if tuple(detector_shapes["WAXS"][:-2]) != leading_shape:
            raise ValueError(f"SAXS/WAXS leading shapes differ: {detector_shapes}")
        if tuple(bsdiodes.shape[:2]) != leading_shape:
            raise ValueError(f"bsdiodes leading shape {bsdiodes.shape[:2]} does not match detector shape {leading_shape}.")

        channel_samples = bsdiodes[..., BSDIODES_CHANNEL]
        valid_count = np.sum(np.isfinite(channel_samples), axis=-1).astype(np.int32)
        mean = np.nanmean(channel_samples, axis=-1)
        std = np.nanstd(channel_samples, axis=-1, ddof=1)
        sem = std / np.sqrt(valid_count)

        count_time_sources = {
            "saxs_count_time": "/entry1/instrument/detector/count_time",
            "waxs_count_time": "/entry1/instrument/Pilatus2M_WAXS/count_time",
        }
        count_times = {}
        for name, path in count_time_sources.items():
            dataset = source[path]
            count_times[name] = (
                _frame_array(dataset[()], leading_shape, name=path),
                str(_decode(dataset.attrs.get("units", "s"))),
            )

        transmission_path = "/entry1/I0/transmission"
        transmission = _frame_array(source[transmission_path][()], leading_shape, name=transmission_path)

    output_file.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_file.with_suffix(output_file.suffix + ".tmp")
    with h5py.File(temporary, "w") as target:
        relative_master = os.path.relpath(master_file, start=output_file.parent)
        target["entry1"] = h5py.ExternalLink(relative_master, "/entry1")
        target.attrs["creator"] = "I22 MoDaCor preprocessing notebook"
        target.attrs["source_file"] = relative_master
        target.attrs["preprocessing_version"] = PREPROCESSING_VERSION

        normalization = target.require_group("/modacor/normalization")
        normalization.attrs["description"] = "Frame-wise arrays reshaped for broadcasting over detector y/x axes."
        normalization.attrs["frame_shape"] = leading_shape
        normalization.attrs["bsdiodes_source"] = "/entry1/bsdiodes/data"
        normalization.attrs["bsdiodes_reduction_axis"] = 2
        normalization.attrs["bsdiodes_channel_index"] = BSDIODES_CHANNEL

        calibration = target.require_group("/modacor/calibration")
        calibration.attrs["description"] = "Scalar calibration values used by the I22 MoDaCor pipelines."
        _write_dataset(
            calibration,
            "absolute_intensity_factor",
            np.asarray(ABSOLUTE_INTENSITY_FACTOR, dtype=float),
            units="dimensionless",
            source="DAWN processing factor for this example",
        )

        _write_dataset(normalization, "bsdiodes_channel_1_mean", _as_detector_divisor(mean), units="dimensionless")
        _write_dataset(normalization, "bsdiodes_channel_1_std", _as_detector_divisor(std), units="dimensionless")
        _write_dataset(normalization, "bsdiodes_channel_1_sem", _as_detector_divisor(sem), units="dimensionless")
        _write_dataset(normalization, "bsdiodes_channel_1_n_valid", _as_detector_divisor(valid_count))
        _write_dataset(normalization, "transmission", _as_detector_divisor(transmission), units="dimensionless")

        for name, (values, units) in count_times.items():
            _write_dataset(normalization, name, _as_detector_divisor(values), units=units)

    temporary.replace(output_file)
    return output_file


# Keep this variable name for the later source-registration cells. These are the original
# DAWN/NeXus calibration files; MoDaCor now resolves their transformation chains directly.
preprocessed_calibration_files = dict(CALIBRATION_FILES)

measurement_inputs = list(sample_files)
if BACKGROUND_FILE.exists() and BACKGROUND_FILE not in measurement_inputs:
    measurement_inputs.append(BACKGROUND_FILE)
preprocessed_measurement_files = {
    source.resolve(): preprocess_i22_measurement(source, overwrite=OVERWRITE_PREPROCESSED)
    for source in measurement_inputs
}
preprocessed_sample_files = [preprocessed_measurement_files[path.resolve()] for path in sample_files]
preprocessed_background_file = preprocessed_measurement_files.get(BACKGROUND_FILE.resolve())

print("Calibration sources:")
for detector, path in preprocessed_calibration_files.items():
    with h5py.File(path, "r") as h5, h5py.File(MASK_FILES[detector], "r") as mask_h5:
        shape = tuple(h5["/entry1/calibration_data/data"].shape)
        mask_shape = tuple(mask_h5["/entry/mask/mask"].shape)
        if shape != mask_shape:
            raise ValueError(f"{detector} calibration shape {shape} does not match mask shape {mask_shape}.")
        detector_path = "/entry1/instrument/detector"
        depends_on = _decode(h5[f"{detector_path}/detector_module/module_offset"].attrs.get("depends_on", ""))
        print(f"  {detector}: {path}  shape={shape}, module_offset depends_on={depends_on!r}")

print(f"Preprocessed {len(preprocessed_sample_files)} sample measurement(s).")
if preprocessed_background_file is None:
    print(f"Background not preprocessed because it is missing: {BACKGROUND_FILE}")
else:
    print(f"Preprocessed background: {preprocessed_background_file}")


## Pipeline Graph Preview

This previews both detector pipelines. It does not require the sample or background files to be registered.


In [ ]:
from modacor.runner.pipeline import Pipeline

pipelines = {}
for detector, pipeline_path in PIPELINE_PATHS.items():
    pipeline = Pipeline.from_yaml_file(yaml_file=pipeline_path)
    pipeline.prepare()
    pipelines[detector] = pipeline
    mermaid_src = pipeline.to_mermaid(direction="TD")
    display(Markdown(f"### {detector}\n\n```mermaid\n{mermaid_src}\n```"))
    print(f"{detector}: {len(pipeline.graph)} configured step(s) from {pipeline_path}")

pipeline = pipelines[DETECTOR]
print(f"Selected pipeline: {DETECTOR} -> {PIPELINE_PATH}")

## Runtime API Helpers

These small helpers keep the HTTP calls readable while still showing which runtime endpoints are used.


In [ ]:
BASE_URLS = {
    detector: f"http://{SERVER_HOST}:{SERVER_PORTS[detector]}"
    for detector in DETECTORS_TO_RUN
}
if not SEPARATE_SERVER_PER_DETECTOR:
    BASE_URLS = {detector: f"http://{SERVER_HOST}:{SERVER_PORT}" for detector in DETECTORS_TO_RUN}
BASE_URL = BASE_URLS[DETECTOR]
SERVER_PROCESSES = {detector: None for detector in DETECTORS_TO_RUN}
SERVER_PROCESS = None


def api_url(path: str, detector: str | None = None) -> str:
    detector = detector or DETECTOR
    return BASE_URLS[detector].rstrip("/") + path


def api_request(
    method: str,
    path: str,
    *,
    detector: str | None = None,
    payload: dict | None = None,
    expected: tuple[int, ...] = (200, 201, 202, 204),
):
    response = requests.request(method, api_url(path, detector=detector), json=payload, timeout=120)
    if response.status_code not in expected:
        message = response.text
        try:
            message = json.dumps(response.json(), indent=2)
        except ValueError:
            pass
        raise RuntimeError(f"{method.upper()} {path} failed with HTTP {response.status_code}:" + "\n" + message)
    if response.status_code == 204 or not response.content:
        return None
    return response.json()


def readiness_ok(detector: str | None = None, timeout: float = 1.0) -> bool:
    try:
        response = requests.get(api_url("/v1/readiness", detector=detector), timeout=timeout)
        return response.ok and bool(response.json().get("ready", False))
    except requests.RequestException:
        return False


def display_text_block(text, *, title=None):
    if title:
        display(Markdown(f"**{title}**"))
    display(Markdown("\n".join(["```text", str(text).replace("```", "`` `"), "```"])))


## Start The Runtime Server

This starts a local MoDaCor server from the same Python environment as the notebook. If a server is already running on the configured port, the notebook reuses it.


In [ ]:
def _server_environment():
    server_env = os.environ.copy()
    if "hdf5plugin" in globals() and hasattr(hdf5plugin, "PLUGINS_PATH"):
        server_env.setdefault("HDF5_PLUGIN_PATH", hdf5plugin.PLUGINS_PATH)
    return server_env


def start_server(detector: str | None = None, timeout_s: float = 45.0):
    global SERVER_PROCESS
    detector = detector or DETECTOR
    base_url = BASE_URLS[detector]
    port = SERVER_PORTS[detector] if SEPARATE_SERVER_PER_DETECTOR else SERVER_PORT
    log_path = SERVER_LOG_PATHS[detector]

    if readiness_ok(detector, timeout=1.0):
        print(f"Runtime server for {detector} is already ready at {base_url}")
        print("If hdf5plugin was just installed, stop and restart this server before processing I22 detector data.")
        return None

    command = [
        sys.executable,
        "-m",
        "modacor.cli",
        "serve",
        "--host",
        SERVER_HOST,
        "--port",
        str(port),
    ]
    print(f"Starting {detector} runtime server:")
    print(" ".join(command))

    server_log_file = open(log_path, "a", buffering=1)

    process = subprocess.Popen(
        command,
        stdout=server_log_file,
        stderr=subprocess.STDOUT,
        env=_server_environment(),
        text=True,
    )
    SERVER_PROCESSES[detector] = process
    if detector == DETECTOR:
        SERVER_PROCESS = process

    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        if readiness_ok(detector, timeout=1.0):
            print(f"Runtime server for {detector} ready at {base_url}")
            return process
        if process.poll() is not None:
            raise RuntimeError(
                f"Runtime server for {detector} exited early with code {process.returncode}. "
                f"See {log_path}."
            )
        time.sleep(0.5)

    raise TimeoutError(f"Runtime server for {detector} did not become ready within {timeout_s:.0f} seconds at {base_url}")


def start_servers(timeout_s: float = 45.0):
    detectors = DETECTORS_TO_RUN if SEPARATE_SERVER_PER_DETECTOR else [DETECTOR]
    return {detector: start_server(detector, timeout_s=timeout_s) for detector in detectors}


def stop_server(detector: str | None = None):
    global SERVER_PROCESS
    detectors = [detector] if detector is not None else list(SERVER_PROCESSES.keys())

    for detector_name in detectors:
        process = SERVER_PROCESSES.get(detector_name)
        if process is None:
            print(f"No notebook-owned {detector_name} server process to stop.")
            continue
        if process.poll() is not None:
            print(f"{detector_name} server process already exited with code {process.returncode}.")
            SERVER_PROCESSES[detector_name] = None
            continue

        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)
        print(f"Stopped notebook-owned {detector_name} runtime server.")
        SERVER_PROCESSES[detector_name] = None
    SERVER_PROCESS = None


atexit.register(stop_server)

start_servers()
readiness = {
    detector: api_request("GET", "/v1/readiness", detector=detector)
    for detector in (DETECTORS_TO_RUN if SEPARATE_SERVER_PER_DETECTOR else [DETECTOR])
}
display(JSON(readiness))


## Create A Fresh Runtime Session

This deletes an existing session with the same ID, then creates a clean one from the current pipeline YAML. Re-run this cell after changing the pipeline YAML or tracer settings.


In [ ]:
def delete_session_if_exists(detector: str, session_id: str):
    response = requests.delete(api_url(f"/v1/sessions/{session_id}", detector=detector), timeout=30)
    if response.status_code == 204:
        print(f"Deleted existing session: {session_id}")
        return
    if response.status_code == 404:
        print(f"No existing session named {session_id!r}.")
        return
    raise RuntimeError(f"DELETE session failed with HTTP {response.status_code}: {response.text}")


def create_server_session(detector: str):
    session_id = SESSION_IDS[detector]
    delete_session_if_exists(detector, session_id)
    session_payload = {
        "session_id": session_id,
        "name": f"I22 {detector} server batch",
        "pipeline": {"yaml_path": str(PIPELINE_PATHS[detector])},
        "trace": {
            "enabled": TRACE_ENABLED,
            "watch": TRACE_WATCH if TRACE_ENABLED else {},
            "record_only_on_change": True,
            "snapshot_processing_data": False,
            "snapshot_step_ids": [],
        },
        "auto_full_reset_on_partial_error": True,
    }
    return api_request("POST", "/v1/sessions", detector=detector, payload=session_payload, expected=(200, 201))


sessions = {detector: create_server_session(detector) for detector in DETECTORS_TO_RUN}
display(JSON(sessions))


## Register Sources

The session receives six explicit source roles: `sample`, `background`, `saxs_calibration`, `saxs_mask`,
`waxs_calibration`, and `waxs_mask`. Only `sample` changes inside the batch loop.


In [ ]:
def build_source_registrations(detector: str, sample_file_for_sample=None):
    background_file = BACKGROUND_FILES[detector]
    sources_to_register = [
        {
            "ref": "saxs_calibration",
            "type": "hdf",
            "location": str(preprocessed_calibration_files["SAXS"]),
        },
        {"ref": "saxs_mask", "type": "hdf", "location": str(MASK_FILES["SAXS"])},
        {
            "ref": "waxs_calibration",
            "type": "hdf",
            "location": str(preprocessed_calibration_files["WAXS"]),
        },
        {"ref": "waxs_mask", "type": "hdf", "location": str(MASK_FILES["WAXS"])},
    ]

    background_source = preprocessed_measurement_files.get(background_file.resolve())
    if background_source is not None:
        sources_to_register.append(
            {"ref": "background", "type": "hdf", "location": str(background_source)}
        )
    else:
        print(f"Background file missing, not registering it yet: {background_file}")

    if sample_file_for_sample is None and preprocessed_sample_files:
        sample_file_for_sample = preprocessed_sample_files[0]
    if sample_file_for_sample is not None:
        sources_to_register.append(
            {"ref": "sample", "type": "hdf", "location": str(sample_file_for_sample)}
        )
    else:
        print("No preprocessed sample file is available yet.")

    return sources_to_register


def register_session_sources(detector: str, sample_file_for_sample=None, *, display_result=True):
    session_id = SESSION_IDS[detector]
    sources_to_register = build_source_registrations(detector, sample_file_for_sample=sample_file_for_sample)
    registered_sources = api_request(
        "PUT",
        f"/v1/sessions/{session_id}/sources",
        detector=detector,
        payload={"sources": sources_to_register},
    )
    if display_result:
        display(JSON(registered_sources))
    return registered_sources


def register_session_plot_sink(detector: str, *, display_result=True):
    session_id = SESSION_IDS[detector]
    registered_sinks = api_request(
        "PUT",
        f"/v1/sessions/{session_id}/sinks",
        detector=detector,
        payload={
            "sinks": [
                {"ref": PLOT_SINK_REF, "type": "plotly_json", "location": "buffer://session"}
            ]
        },
    )
    if display_result:
        display(JSON(registered_sinks))
    return registered_sinks


def register_session_result_hdf_sink(detector: str, output_path: Path, *, display_result=True):
    session_id = SESSION_IDS[detector]
    registered_sink = api_request(
        "POST",
        f"/v1/sessions/{session_id}/sinks/patch",
        detector=detector,
        payload={
            "ref": RESULT_HDF_SINK_REF,
            "type": "hdf",
            "location": str(output_path),
        },
    )
    if display_result:
        display(JSON(registered_sink))
    return registered_sink


def reset_session_after_failed_sample(detector: str):
    print(f"Resetting the {detector} server session after a failed sample because rollback snapshots are disabled.")
    session = create_server_session(detector)
    registered_sources = register_session_sources(detector, display_result=False)
    registered_sinks = register_session_plot_sink(detector, display_result=False)
    print(f"{detector} session reset complete. The next sample will run in full mode to seed fresh state.")
    return {"session": session, "sources": registered_sources, "sinks": registered_sinks}


registered_sources = {
    detector: register_session_sources(detector, display_result=False)
    for detector in DETECTORS_TO_RUN
}
registered_plot_sinks = {
    detector: register_session_plot_sink(detector, display_result=False)
    for detector in DETECTORS_TO_RUN
}
display(JSON({"sources": registered_sources, "plot_sinks": registered_plot_sinks}))


## Session Status

This optional check shows the current session, registered sources, and trace configuration.


In [ ]:
session_status = {
    detector: api_request("GET", f"/v1/sessions/{SESSION_IDS[detector]}", detector=detector)
    for detector in DETECTORS_TO_RUN
}
display(JSON(session_status))


## Live Server Plots

These links open auto-refreshing Plotly pages served by the MoDaCor runtime. They show the latest plot payload published by the pipeline after each processed sample.


In [ ]:
def live_plot_url(detector: str, plot_id: str) -> str:
    session_id = SESSION_IDS[detector]
    return api_url(f"/v1/sessions/{session_id}/plots/{PLOT_SINK_REF}/{plot_id}", detector=detector)


def display_live_plot_links():
    lines = []
    for detector in DETECTORS_TO_RUN:
        plot_ids = LIVE_PLOT_IDS[detector]
        lines.append(f"### {detector}")
        lines.append(f"- [1D corrected I(Q)]({live_plot_url(detector, plot_ids['1d'])})")
        lines.append(f"- [2D corrected detector image]({live_plot_url(detector, plot_ids['2d'])})")
    display(Markdown("\n".join(lines)))


display_live_plot_links()


## Batch Processing Loop

The first processed sample runs in `full` mode to seed runtime state. Later samples update only the `sample` source and run in `auto` mode, so the server reuses unchanged state where possible.

With `ROLLBACK_SNAPSHOT = False`, failed partial runs are handled by recreating the session before continuing. The next sample then runs in `full` mode once to seed clean state again.


In [ ]:
# limit number of sample files to the first 10 for testing purposes only:
sample_files = sample_files[:10]
preprocessed_sample_files = preprocessed_sample_files[:10]

In [ ]:
run_results = []
failed_results = []
session_has_processing_state = {detector: False for detector in DETECTORS_TO_RUN}


def process_detector_sample(detector: str, index: int, sample_file: Path, preprocessed_sample_file: Path):
    session_id = SESSION_IDS[detector]
    run_name = f"{sample_file.stem}_{detector.lower()}"
    output_path = OUTPUT_DIR / f"{sample_file.stem}_{detector.lower()}_server_result.h5"
    mode = "auto" if session_has_processing_state[detector] else "full"

    try:
        api_request(
            "PUT",
            f"/v1/sessions/{session_id}/sources",
            detector=detector,
            payload={
                "sources": [
                    {"ref": "sample", "type": "hdf", "location": str(preprocessed_sample_file)}
                ]
            },
        )

        register_session_result_hdf_sink(detector, output_path, display_result=False)

        process_payload = {
            "mode": mode,
            "run_name": run_name,
            "rollback_snapshot": ROLLBACK_SNAPSHOT,
            "write_hdf": {
                "path": str(output_path),
                "data_paths": SAMPLE_OUTPUT_DATA_PATHS,
            },
        }
        if mode == "auto":
            process_payload["changed_sources"] = ["sample"]

        result = api_request(
            "POST",
            f"/v1/sessions/{session_id}/process",
            detector=detector,
            payload=process_payload,
        )
    except Exception as exc:
        latest_error = None
        diagnostics_error = None
        try:
            latest_error = api_request("GET", f"/v1/sessions/{session_id}/errors/latest", detector=detector)
        except Exception as diagnostics_exc:
            diagnostics_error = str(diagnostics_exc)

        return {
            "ok": False,
            "detector": detector,
            "index": index,
            "sample": str(sample_file),
            "preprocessed_sample": str(preprocessed_sample_file),
            "output": str(output_path),
            "mode": mode,
            "error": str(exc),
            "latest_error": latest_error,
            "diagnostics_error": diagnostics_error,
        }

    return {
        "ok": True,
        "detector": detector,
        "index": index,
        "sample": str(sample_file),
        "preprocessed_sample": str(preprocessed_sample_file),
        "output": str(output_path),
        "mode": mode,
        "result": result,
    }


def process_sample_for_detectors(index: int, sample_file: Path, preprocessed_sample_file: Path):
    if PARALLEL_DETECTOR_RUNS and len(DETECTORS_TO_RUN) > 1:
        with ThreadPoolExecutor(max_workers=len(DETECTORS_TO_RUN)) as executor:
            futures = {
                executor.submit(process_detector_sample, detector, index, sample_file, preprocessed_sample_file): detector
                for detector in DETECTORS_TO_RUN
            }
            results = [future.result() for future in as_completed(futures)]
        return sorted(results, key=lambda item: DETECTORS_TO_RUN.index(item["detector"]))

    return [
        process_detector_sample(detector, index, sample_file, preprocessed_sample_file)
        for detector in DETECTORS_TO_RUN
    ]


if not sample_files:
    print("No sample files found yet. Rerun discovery and preprocessing after copying finishes.")
else:
    missing_backgrounds = [detector for detector in DETECTORS_TO_RUN if BACKGROUND_FILES[detector].resolve() not in preprocessed_measurement_files]
    if missing_backgrounds:
        missing = {detector: str(BACKGROUND_FILES[detector]) for detector in missing_backgrounds}
        raise FileNotFoundError(f"Background file(s) missing: {missing}")

    sample_batch = list(zip(sample_files, preprocessed_sample_files, strict=True))
    if MAX_SAMPLES_TO_PROCESS is not None:
        sample_batch = sample_batch[:MAX_SAMPLES_TO_PROCESS]
    print(f"Detector concurrency: {'on' if PARALLEL_DETECTOR_RUNS else 'off'}")
    print(f"Server mode: {'separate detector servers' if SEPARATE_SERVER_PER_DETECTOR else 'single shared server'}")
    print(f"Processing {len(sample_batch)} of {len(sample_files)} discovered sample file(s).")
    for index, (sample_file, preprocessed_sample_file) in enumerate(sample_batch):
        print(f"\n[{index + 1}/{len(sample_batch)}] {sample_file.name}")
        print(f"Source: {preprocessed_sample_file}")

        sample_results = process_sample_for_detectors(index, sample_file, preprocessed_sample_file)
        sample_failures = []
        for item in sample_results:
            detector = item["detector"]
            if item["ok"]:
                session_has_processing_state[detector] = True
                run_results.append({key: value for key, value in item.items() if key != "ok"})
                result = item["result"]
                print(
                    f"{detector}: {result.get('status')} mode={item['mode']} "
                    f"effective_mode={result.get('effective_mode')} run_id={result.get('run_id')}"
                )
                continue

            sample_failures.append(item)
            failed_results.append({key: value for key, value in item.items() if key != "ok"})
            print(f"{detector}: failed mode={item['mode']} output={item['output']}")
            print(item["error"])
            if item.get("diagnostics_error"):
                print("Could not fetch latest server diagnostics:", item["diagnostics_error"])
            if item.get("latest_error"):
                display(JSON(item["latest_error"]))

        if sample_failures:
            failed_server_down = any(not readiness_ok(item["detector"], timeout=2.0) for item in sample_failures)
            if not CONTINUE_ON_SAMPLE_ERROR or failed_server_down:
                failed = ", ".join(item["detector"] for item in sample_failures)
                raise RuntimeError(f"Detector run(s) failed for {sample_file.name}: {failed}")

            if RESET_SESSION_AFTER_FAILURE:
                for item in sample_failures:
                    reset_session_after_failed_sample(item["detector"])
                    session_has_processing_state[item["detector"]] = False
            print("Continuing with the next sample.")

summary = {"succeeded": run_results, "failed": failed_results}
print(f"Succeeded: {len(run_results)}  Failed: {len(failed_results)}")
display(JSON(summary))


## Optional Diagnostics

Run this only when something looks wrong. Trace reports are available only when `TRACE_ENABLED = True` before creating the session.


In [ ]:
for detector in DETECTORS_TO_RUN:
    session_id = SESSION_IDS[detector]
    display(Markdown(f"### {detector}"))
    if TRACE_ENABLED:
        runs_payload = api_request("GET", f"/v1/sessions/{session_id}/runs", detector=detector)
        runs = runs_payload.get("runs", [])
        latest_run = runs[-1] if runs else None
        trace_report = latest_run.get("trace_report") if latest_run else None

        if not trace_report:
            latest_error = api_request("GET", f"/v1/sessions/{session_id}/errors/latest", detector=detector)
            error_payload = latest_error.get("current_error") or latest_error.get("latest_error") or {}
            trace_report = (error_payload.get("details") or {}).get("trace_report")

        if trace_report:
            display_text_block(trace_report, title="Tracer report")
        else:
            print("No trace report recorded for this session yet.")
    else:
        print("Tracing is disabled. Set TRACE_ENABLED = True and recreate the session to collect trace reports.")

    latest_error = api_request("GET", f"/v1/sessions/{session_id}/errors/latest", detector=detector)
    error_payload = latest_error.get("current_error") or latest_error.get("latest_error") or {}
    error_details = error_payload.get("details") or {}

    if error_payload:
        display(JSON(latest_error))
        if error_details.get("traceback"):
            display_text_block(error_details["traceback"], title="Traceback")
    else:
        print("No server error is currently recorded for this session.")


## Stop The Server

Run this when you are done. The notebook also registers an automatic cleanup hook for a server process it started itself.


In [ ]:
stop_server()


## Example 1 — Local Chunked HDF5 Assembly Validation

This first real-data chunking checkpoint isolates storage behavior from the correction pipeline. It creates a zero-copy HDF5 virtual view over a configurable frame range from an actual I22 detector dataset, writes that view once with the ordinary `HDFProcessingSink`, then assembles the same values through `HDFChunkedProcessingSink` in several frame chunks. The two stored signals are compared exactly and the chunk benchmark records timing, layout, and process peak RSS.

The default ten-frame view keeps the ordinary baseline manageable; set `CHUNK_VALIDATION_FRAME_SLICE = slice(None)` to exercise all frames. Generated files stay below ignored `work/chunk_validation/`. The virtual view contains no copied detector data.

This does **not** yet claim whole-pipeline equivalence. The I22 correction pipelines reduce the frame dimensions before later corrections and integration, so independently averaging already-reduced chunk results would be incorrect. Pipeline-level chunk validation needs an explicit reducer boundary or a comparison of independent per-chunk pipeline runs.


In [ ]:
from modacor import ureg
from modacor.dataclasses.basedata import BaseData
from modacor.dataclasses.databundle import DataBundle
from modacor.dataclasses.processing_data import ProcessingData
from modacor.io.hdf import HDFProcessingSink

CHUNK_VALIDATION_DETECTOR = DETECTOR
CHUNK_VALIDATION_FRAME_SLICE = slice(0, 10)
CHUNK_VALIDATION_CHUNK_SIZE = 2
CHUNK_VALIDATION_COMPRESSION = "gzip"
CHUNK_VALIDATION_COMPRESSION_LEVEL = 1
CHUNK_VALIDATION_DATASETS = {
    "SAXS": "/entry1/detector/data",
    "WAXS": "/entry1/Pilatus2M_WAXS/data",
}


def create_i22_frame_view(source_path, dataset_path, output_path, frame_slice):
    source_path = Path(source_path).resolve()
    output_path = Path(output_path).resolve()
    with h5py.File(source_path, "r") as source_h5:
        source_dataset = source_h5[dataset_path]
        source_shape = tuple(source_dataset.shape)
        source_dtype = source_dataset.dtype

    start, stop, stride = frame_slice.indices(source_shape[1])
    if stride != 1:
        raise ValueError("This initial I22 validation requires a contiguous frame slice.")
    frame_count = stop - start
    if frame_count < 1:
        raise ValueError("CHUNK_VALIDATION_FRAME_SLICE selects no frames.")

    view_shape = list(source_shape)
    view_shape[1] = frame_count
    layout = h5py.VirtualLayout(shape=tuple(view_shape), dtype=source_dtype)
    virtual_source = h5py.VirtualSource(str(source_path), dataset_path, shape=source_shape)
    layout[...] = virtual_source[:, start:stop, ...]

    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    if temporary.exists():
        temporary.unlink()
    with h5py.File(temporary, "w", libver="latest") as output_h5:
        parent_path, dataset_name = dataset_path.rsplit("/", 1)
        parent = output_h5.require_group(parent_path)
        parent.create_virtual_dataset(dataset_name, layout)
        output_h5.attrs["source_file"] = str(source_path)
        output_h5.attrs["source_dataset"] = dataset_path
        output_h5.attrs["source_frame_slice"] = f"{start}:{stop}"
    temporary.replace(output_path)
    return output_path, tuple(view_shape)


def write_ordinary_identity_baseline(source_path, dataset_path, output_path):
    with h5py.File(source_path, "r") as source_h5:
        values = np.asarray(source_h5[dataset_path][()])
    signal = BaseData(signal=values, units=ureg.dimensionless, rank_of_data=2)
    bundle = DataBundle({"signal": signal})
    bundle.default_plot = "signal"
    processing_data = ProcessingData({"sample": bundle})
    output_path = Path(output_path)
    if output_path.exists():
        output_path.unlink()
    HDFProcessingSink(resource_location=output_path).write(
        "baseline", processing_data, data_paths=["/sample/signal"]
    )
    del processing_data, bundle, signal, values
    return output_path


def run_chunked_identity_benchmark(source_path, dataset_path, output_path, report_path):
    benchmark_script = Path(modacor.__file__).resolve().parents[2] / "scripts" / "benchmark_chunked_hdf.py"
    if not benchmark_script.is_file():
        raise FileNotFoundError(f"Chunk benchmark script not found: {benchmark_script}")
    with h5py.File(source_path, "r") as source_h5:
        shape = tuple(source_h5[dataset_path].shape)
    storage_chunks = (1, 1, min(256, shape[-2]), min(256, shape[-1]))
    command = [
        sys.executable,
        str(benchmark_script),
        "--source-hdf",
        str(source_path),
        "--source-dataset",
        dataset_path.strip("/"),
        "--output",
        str(output_path),
        "--report",
        str(report_path),
        "--chunk-axis",
        "1",
        "--chunk-size",
        str(CHUNK_VALIDATION_CHUNK_SIZE),
        "--rank-of-data",
        "2",
        "--storage-chunks",
        ",".join(str(value) for value in storage_chunks),
        "--compression",
        CHUNK_VALIDATION_COMPRESSION,
        "--overwrite",
    ]
    if CHUNK_VALIDATION_COMPRESSION == "gzip" and CHUNK_VALIDATION_COMPRESSION_LEVEL is not None:
        command.extend(["--compression-level", str(CHUNK_VALIDATION_COMPRESSION_LEVEL)])
    completed = subprocess.run(command, capture_output=True, text=True)
    if completed.returncode != 0:
        raise RuntimeError(
            "Chunk benchmark failed:\n" + (completed.stderr.strip() or completed.stdout.strip())
        )
    return json.loads(completed.stdout)


if not sample_files:
    raise FileNotFoundError("No I22 sample master is available for chunk validation.")
if CHUNK_VALIDATION_DETECTOR not in CHUNK_VALIDATION_DATASETS:
    raise ValueError(f"Unknown chunk-validation detector: {CHUNK_VALIDATION_DETECTOR}")

chunk_validation_dir = WORK_DIR / "chunk_validation"
chunk_validation_dir.mkdir(parents=True, exist_ok=True)
chunk_validation_source = sample_files[0]
chunk_validation_dataset = CHUNK_VALIDATION_DATASETS[CHUNK_VALIDATION_DETECTOR]
chunk_validation_view = chunk_validation_dir / f"{chunk_validation_source.stem}_{CHUNK_VALIDATION_DETECTOR.lower()}_view.h5"
ordinary_identity_output = chunk_validation_dir / f"{chunk_validation_source.stem}_{CHUNK_VALIDATION_DETECTOR.lower()}_ordinary.h5"
chunked_identity_output = chunk_validation_dir / f"{chunk_validation_source.stem}_{CHUNK_VALIDATION_DETECTOR.lower()}_chunked.h5"
chunked_identity_report = chunk_validation_dir / f"{chunk_validation_source.stem}_{CHUNK_VALIDATION_DETECTOR.lower()}_chunked.json"

chunk_validation_view, chunk_validation_shape = create_i22_frame_view(
    chunk_validation_source,
    chunk_validation_dataset,
    chunk_validation_view,
    CHUNK_VALIDATION_FRAME_SLICE,
)
write_ordinary_identity_baseline(
    chunk_validation_view, chunk_validation_dataset, ordinary_identity_output
)
chunk_validation_report = run_chunked_identity_benchmark(
    chunk_validation_view,
    chunk_validation_dataset,
    chunked_identity_output,
    chunked_identity_report,
)
print(f"Source: {chunk_validation_source.name}::{chunk_validation_dataset}")
print(f"Virtual view shape: {chunk_validation_shape}")
print(f"Ordinary output: {ordinary_identity_output}")
print(f"Chunked output: {chunked_identity_output}")


### Compare Ordinary And Chunked Storage

Comparison is performed one frame chunk at a time so this verification step does not load both assembled outputs completely into memory.


In [ ]:
ordinary_signal_path = "/processing/result/baseline/sample/signal/signal"
chunked_signal_path = "/processing/result/benchmark/sample/signal/signal"

with (
    h5py.File(ordinary_identity_output, "r") as ordinary_h5,
    h5py.File(chunked_identity_output, "r") as chunked_h5,
):
    ordinary_signal = ordinary_h5[ordinary_signal_path]
    chunked_signal = chunked_h5[chunked_signal_path]
    if ordinary_signal.shape != chunked_signal.shape or ordinary_signal.dtype != chunked_signal.dtype:
        raise AssertionError(
            f"Stored schema differs: {ordinary_signal.shape}/{ordinary_signal.dtype} versus "
            f"{chunked_signal.shape}/{chunked_signal.dtype}"
        )
    for start in range(0, ordinary_signal.shape[1], CHUNK_VALIDATION_CHUNK_SIZE):
        stop = min(start + CHUNK_VALIDATION_CHUNK_SIZE, ordinary_signal.shape[1])
        np.testing.assert_array_equal(
            ordinary_signal[:, start:stop, ...],
            chunked_signal[:, start:stop, ...],
        )

comparison_summary = {
    "status": "exact match",
    "shape": list(chunk_validation_shape),
    "dtype": chunk_validation_report["dtype"],
    "chunk_count": chunk_validation_report["chunk_count"],
    "chunk_size": chunk_validation_report["chunk_size"],
    "storage_layout": chunk_validation_report["storage_layout"],
    "timing_s": chunk_validation_report["timing_s"],
    "memory": chunk_validation_report["memory"],
}
display(JSON(comparison_summary))


## Example 2 — BufferSource Processing Into A Chunked HDF5 Result

This example processes both SAXS and WAXS for all four sample measurements as ten chunks of ten frames each. Each detector-specific server session receives changing sample arrays through a `BufferSource`; its background, calibration, and mask sources remain registered as HDF5 sources and are reused by partial reruns. A pilot chunk establishes each detector's output schema, after which the notebook initializes, fills, inspects, and finalizes two independent 40-chunk outputs. Both detector runs are stored in one physical HDF5 file under `processed_saxs_frame_chunks` and `processed_waxs_frame_chunks`.

Background handling is deliberately asymmetric: each detector's pilot full run reads and reduces its complete background once, while subsequent partial runs declare only the `sample` source as changed and reuse that reduced background. Background frames are therefore not matched to sample chunk boundaries. The sample and background may contain different numbers of frames because each branch independently averages axes 0 and 1 before subtraction; their detector shapes after reduction must still be compatible, and each source's normalization arrays must match or broadcast to that source's own leading dimensions. This is an aggregate-background strategy, not time- or frame-paired subtraction. It bounds sample-chunk memory, but the complete background working set must fit in memory. The detector sessions run sequentially and are deleted after finalization so the SAXS and WAXS background working sets do not need to coexist.

The stored signal therefore has leading dimensions `(measurement, frame_chunk, ...)`: each element is the corrected result obtained by reducing one ten-frame input chunk. This is intentionally not presented as a replacement for reducing all 100 frames together. Combining chunk-level reductions into that single result requires a mergeable reducer and is a separate numerical-validation step.

The plotting and intermediate sink steps are removed from an in-memory copy of the validation pipeline to avoid serializing large detector images for every chunk. All numerical correction and integration steps are retained. The original tracked YAML is not modified.

Lightweight, array-free trace events are enabled. Every successful publication stores its events beneath `/processing/tracer/<run_name>/chunks/<chunk_id>/`; processing-data snapshots remain disabled.


In [ ]:
# Configure the server-run example and define its upload, plan, and placement helpers.
# The helpers keep sample detector data and per-frame normalization arrays on identical slices.
from io import BytesIO

import yaml

from modacor.io.chunking import (
    AxisSelector,
    ChunkArrayLayout,
    ChunkOutputLayout,
    ChunkPlacement,
    ChunkPlan,
    ChunkSourceBinding,
    ChunkSpec,
    PlacementBinding,
)

CHUNK_SERVER_DETECTORS = tuple(DETECTORS_TO_RUN)
CHUNK_SERVER_DATASETS = {
    "SAXS": "/entry1/detector/data",
    "WAXS": "/entry1/Pilatus2M_WAXS/data",
}
CHUNK_SERVER_CHUNK_SIZE = 10
CHUNK_SERVER_FRAMES_PER_MEASUREMENT = 100
CHUNK_SERVER_MEASUREMENT_LIMIT = 4
CHUNK_SERVER_COMPRESSION = "gzip"
CHUNK_SERVER_COMPRESSION_LEVEL = 1
CHUNK_SERVER_STOP_SERVER_AFTER_RUN = True
CHUNK_SERVER_DELETE_SESSION_AFTER_RUN = True
CHUNK_SERVER_TRACE_ENABLED = True


def chunk_validation_pipeline_yaml(detector):
    # Retain numerical processing while omitting plots and ordinary intermediate files.
    pipeline_spec = yaml.safe_load(PIPELINE_PATHS[detector].read_text(encoding="utf-8"))
    steps = pipeline_spec["steps"]
    for step_id in ("PL_IQ", "SV_IQ", "PL_2D", "SV_2D"):
        steps.pop(step_id, None)
    steps["AV"]["requires_steps"] = ["PO"]
    pipeline_spec["name"] = f"{pipeline_spec['name']} - chunk validation"
    pipeline_spec["description"] = (
        f"{pipeline_spec.get('description', '')} Test-only copy without visualization or intermediate sinks."
    ).strip()
    return yaml.safe_dump(pipeline_spec, sort_keys=False)


def put_buffer_array(detector, session_id, source_ref, data_key, values):
    payload = BytesIO()
    np.save(payload, np.asarray(values), allow_pickle=False)
    response = requests.put(
        api_url(
            f"/v1/sessions/{session_id}/buffers/sources/{source_ref}/arrays/{data_key.strip('/')}",
            detector=detector,
        ),
        data=payload.getvalue(),
        headers={"Content-Type": "application/x-npy"},
        timeout=120,
    )
    if response.status_code != 200:
        raise RuntimeError(f"Buffer upload failed with HTTP {response.status_code}: {response.text}")
    return response.json()


def put_buffer_attrs(detector, session_id, source_ref, data_key, attrs):
    return api_request(
        "PUT",
        f"/v1/sessions/{session_id}/buffers/sources/{source_ref}/attrs/{data_key.strip('/')}",
        detector=detector,
        payload=attrs,
    )


def upload_i22_sample_chunk(detector, session_id, source_path, start, stop):
    # Stage one sample slice plus every sample-side value aligned with its frames.
    detector_path = CHUNK_SERVER_DATASETS[detector]
    count_time_name = f"{detector.lower()}_count_time"
    aligned_paths = [
        detector_path,
        "/modacor/normalization/bsdiodes_channel_1_mean",
        "/modacor/normalization/bsdiodes_channel_1_std",
        f"/modacor/normalization/{count_time_name}",
    ]
    selection = (slice(None), slice(start, stop), slice(None), slice(None))
    with h5py.File(source_path, "r") as source_h5:
        for data_path in aligned_paths:
            put_buffer_array(
                detector, session_id, "sample", data_path, source_h5[data_path][selection]
            )
        calibration_path = "/modacor/calibration/absolute_intensity_factor"
        put_buffer_array(
            detector, session_id, "sample", calibration_path, source_h5[calibration_path][()]
        )
        put_buffer_attrs(
            detector,
            session_id,
            "sample",
            "/modacor/normalization/bsdiodes_channel_1_mean",
            {"units": str(_decode(source_h5["/modacor/normalization/bsdiodes_channel_1_mean"].attrs["units"]))},
        )
        put_buffer_attrs(
            detector,
            session_id,
            "sample",
            f"/modacor/normalization/{count_time_name}",
            {"units": str(_decode(source_h5[f"/modacor/normalization/{count_time_name}"].attrs["units"]))},
        )
        put_buffer_attrs(
            detector,
            session_id,
            "sample",
            calibration_path,
            {"units": str(_decode(source_h5[calibration_path].attrs["units"]))},
        )


def _hdf_text(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


def sample_aligned_data_paths(detector):
    # These four arrays share the detector's measurement/frame axes and must receive one slice.
    return (
        CHUNK_SERVER_DATASETS[detector],
        "/modacor/normalization/bsdiodes_channel_1_mean",
        "/modacor/normalization/bsdiodes_channel_1_std",
        f"/modacor/normalization/{detector.lower()}_count_time",
    )


def build_server_chunk_plan(
    pilot_path, pilot_run_name, detector, measurement_files, source_shape, *, source_mode="buffer"
):
    # Derive the stored array schema from a real pilot result, then describe all placements.
    if source_mode not in {"buffer", "hdf", "tiled"}:
        raise ValueError("source_mode must be 'buffer', 'hdf', or 'tiled'.")
    chunks_per_measurement = int(np.ceil(CHUNK_SERVER_FRAMES_PER_MEASUREMENT / CHUNK_SERVER_CHUNK_SIZE))
    with h5py.File(pilot_path, "r") as pilot_h5:
        group = pilot_h5[f"/processing/result/{pilot_run_name}/sample/signal"]
        signal_dataset = group["signal"]
        chunk_result_shape = tuple(signal_dataset.shape)
        final_shape = (len(measurement_files), chunks_per_measurement, *chunk_result_shape)
        arrays = [ChunkArrayLayout("signal", final_shape, signal_dataset.dtype.str)]

        if "weights" in group:
            weights = group["weights"]
            if tuple(weights.shape) != chunk_result_shape:
                raise ValueError(f"Pilot weights shape {weights.shape} does not follow signal {chunk_result_shape}.")
            arrays.append(ChunkArrayLayout("weights", final_shape, weights.dtype.str))
        elif float(group.attrs.get("weight_scalar", 1.0)) != 1.0:
            weight = np.asarray(group.attrs["weight_scalar"])
            arrays.append(
                ChunkArrayLayout(
                    "weights", (), weight.dtype.str, PlacementBinding(kind="static")
                )
            )

        uncertainties = group.get("uncertainties")
        if uncertainties is not None:
            for name, dataset in uncertainties.items():
                if tuple(dataset.shape) != chunk_result_shape:
                    raise ValueError(
                        f"Pilot uncertainty {name!r} shape {dataset.shape} does not follow signal."
                    )
                arrays.append(
                    ChunkArrayLayout(f"uncertainties/{name}", final_shape, dataset.dtype.str)
                )

        pilot_axis_names = tuple(_hdf_text(value) for value in group.attrs.get("axes", ()))
        for axis_name in dict.fromkeys(name for name in pilot_axis_names if name != "."):
            axis = group[axis_name]
            arrays.append(
                ChunkArrayLayout(
                    f"axes/{axis_name}",
                    tuple(axis.shape),
                    axis.dtype.str,
                    PlacementBinding(kind="static"),
                    units=_hdf_text(axis.attrs["units"]),
                    rank_of_data=int(axis.attrs["rank_of_data"]),
                )
            )

        output = ChunkOutputLayout(
            output_id="sample_signal",
            processing_path="/sample/signal",
            destination_path="sample/signal",
            units=_hdf_text(signal_dataset.attrs["units"]),
            rank_of_data=int(signal_dataset.attrs["rank_of_data"]),
            arrays=tuple(arrays),
            axis_names=(".", ".", *pilot_axis_names),
        )

    chunk_ids = tuple(
        f"m{measurement_index:03d}-c{chunk_index:03d}"
        for measurement_index in range(len(measurement_files))
        for chunk_index in range(chunks_per_measurement)
    )
    driver = {
        "source_dataset": CHUNK_SERVER_DATASETS[detector],
        "full_shape": list(source_shape),
        "measurement_files": [str(path) for path in measurement_files],
        "source_mode": source_mode,
    }
    source_bindings = ()
    if source_mode in {"hdf", "tiled"}:
        # Direct sources are sliced by the runtime; BufferSource values arrive pre-sliced.
        driver["source"] = f"sample::{CHUNK_SERVER_DATASETS[detector]}"
        source_bindings = tuple(
            ChunkSourceBinding("sample", data_path, "aligned")
            for data_path in sample_aligned_data_paths(detector)
        )

    return ChunkPlan(
        schema_version="1.0",
        plan_id=(
            f"i22-{detector.lower()}-{len(measurement_files)}m-"
            f"{CHUNK_SERVER_FRAMES_PER_MEASUREMENT}f-{CHUNK_SERVER_CHUNK_SIZE}f"
            f"{'-' + source_mode if source_mode != 'buffer' else ''}-chunks"
        ),
        total_chunks=len(chunk_ids),
        expected_chunk_ids=chunk_ids,
        outputs=(output,),
        driver=driver,
        batch_axes=(0, 1),
        data_axes=(2, 3),
        axis_rules=(
            {
                "axis": 1,
                "start": 0,
                "stop": CHUNK_SERVER_FRAMES_PER_MEASUREMENT,
                "stride": 1,
                "chunk_size": CHUNK_SERVER_CHUNK_SIZE,
            },
        ),
        bindings=(
            {"source": "sample detector and normalization arrays", "role": "aligned"},
            {"source": "background, calibration, and masks", "role": "static"},
        ),
        source_bindings=source_bindings,
    )


def server_chunk_spec(plan, measurement_index, chunk_index, start, stop):
    # Bind one source-frame interval to one (measurement, chunk) destination position.
    source_selection = (
        AxisSelector.all(),
        AxisSelector.sliced(start, stop),
        AxisSelector.all(),
        AxisSelector.all(),
    )
    output = plan.output("sample_signal")
    destination_selection = (
        AxisSelector.index(measurement_index),
        AxisSelector.index(chunk_index),
        *(AxisSelector.all() for _ in output.signal.final_shape[2:]),
    )
    ordinal = measurement_index * int(
        np.ceil(CHUNK_SERVER_FRAMES_PER_MEASUREMENT / CHUNK_SERVER_CHUNK_SIZE)
    ) + chunk_index
    expected_input_shape = list(plan.driver["full_shape"])
    expected_input_shape[1] = stop - start
    return ChunkSpec(
        schema_version=plan.schema_version,
        plan_id=plan.plan_id,
        plan_hash=plan.plan_hash,
        chunk_id=plan.expected_chunk_ids[ordinal],
        ordinal=ordinal,
        grid_index=(measurement_index, chunk_index),
        source_selection=source_selection,
        expected_input_shape=tuple(expected_input_shape),
        placements=(
            ChunkPlacement(
                output_id=output.output_id,
                destination_selection=destination_selection,
                expected_shape=tuple(output.signal.final_shape[2:]),
            ),
        ),
    )


In [ ]:
# Validate the shared measurements and each detector shape before starting either session.
if not CHUNK_SERVER_DETECTORS:
    raise ValueError("CHUNK_SERVER_DETECTORS must contain at least one detector.")
unknown_chunk_detectors = set(CHUNK_SERVER_DETECTORS) - set(DETECTORS_TO_RUN)
if unknown_chunk_detectors:
    raise ValueError(f"Chunk detectors are not configured: {sorted(unknown_chunk_detectors)}")
if CHUNK_SERVER_CHUNK_SIZE < 1:
    raise ValueError("CHUNK_SERVER_CHUNK_SIZE must be positive.")

chunk_server_measurements = list(
    zip(sample_files, preprocessed_sample_files, strict=True)
)[:CHUNK_SERVER_MEASUREMENT_LIMIT]
if not chunk_server_measurements:
    raise FileNotFoundError("No preprocessed I22 sample measurements are available.")

chunk_server_source_shapes = {}
for detector in CHUNK_SERVER_DETECTORS:
    detector_path = CHUNK_SERVER_DATASETS[detector]
    detector_shapes = []
    for _master_path, source_path in chunk_server_measurements:
        with h5py.File(source_path, "r") as source_h5:
            detector_shapes.append(tuple(source_h5[detector_path].shape))
    if len(set(detector_shapes)) != 1:
        raise ValueError(f"{detector} sample detector shapes differ: {detector_shapes}")
    source_shape = detector_shapes[0]
    if CHUNK_SERVER_FRAMES_PER_MEASUREMENT > source_shape[1]:
        raise ValueError(
            f"{detector} requested {CHUNK_SERVER_FRAMES_PER_MEASUREMENT} frames, "
            f"but the source has {source_shape[1]}."
        )
    chunk_server_source_shapes[detector] = source_shape

# Both detector-specific plans append to this file under distinct run names.
chunk_server_dir = WORK_DIR / "chunk_server"
chunk_server_dir.mkdir(parents=True, exist_ok=True)
chunk_server_output_path = chunk_server_dir / (
    f"i22_saxs_waxs_{len(chunk_server_measurements)}x"
    f"{CHUNK_SERVER_FRAMES_PER_MEASUREMENT}_frame_chunks.h5"
)
print(
    f"Prepared {len(chunk_server_measurements)} measurements for "
    f"{', '.join(CHUNK_SERVER_DETECTORS)} -> {chunk_server_output_path}"
)


In [ ]:
# Run one complete detector lifecycle at a time so its background state can be released.
def process_chunked_detector(detector):
    start_server(detector)
    session_id = f"i22-{detector.lower()}-frame-chunks"
    delete_session_if_exists(detector, session_id)

    try:
        # Create a trace-enabled session and keep only the changing sample in a buffer.
        api_request(
            "POST",
            "/v1/sessions",
            detector=detector,
            payload={
                "session_id": session_id,
                "name": f"I22 {detector} ten-frame chunk validation",
                "pipeline": {"yaml_text": chunk_validation_pipeline_yaml(detector)},
                "trace": {
                    "enabled": CHUNK_SERVER_TRACE_ENABLED,
                    "watch": TRACE_WATCH,
                },
                "auto_full_reset_on_partial_error": True,
            },
        )
        chunk_sources = build_source_registrations(detector)
        chunk_sources = [source for source in chunk_sources if source["ref"] != "sample"]
        chunk_sources.append({"ref": "sample", "type": "buffer", "location": "buffer://session"})
        api_request(
            "PUT",
            f"/v1/sessions/{session_id}/sources",
            detector=detector,
            payload={"sources": chunk_sources},
        )

        # The pilot computes the complete background branch and fixes this detector's output schema.
        first_master, first_source = chunk_server_measurements[0]
        pilot_stop = min(CHUNK_SERVER_CHUNK_SIZE, CHUNK_SERVER_FRAMES_PER_MEASUREMENT)
        upload_i22_sample_chunk(detector, session_id, first_source, 0, pilot_stop)
        pilot_run_name = f"{first_master.stem}_{detector.lower()}_pilot"
        pilot_output_path = chunk_server_dir / f"{pilot_run_name}.h5"
        if pilot_output_path.exists():
            pilot_output_path.unlink()
        api_request(
            "POST",
            f"/v1/sessions/{session_id}/process",
            detector=detector,
            payload={
                "mode": "full",
                "run_name": pilot_run_name,
                "rollback_snapshot": False,
                "write_hdf": {
                    "path": str(pilot_output_path),
                    "data_paths": ["/sample/signal"],
                },
            },
        )

        server_plan = build_server_chunk_plan(
            pilot_output_path,
            pilot_run_name,
            detector,
            [master for master, _source in chunk_server_measurements],
            chunk_server_source_shapes[detector],
        )
        run_subpath = f"processed_{detector.lower()}_frame_chunks"
        initialized_output = api_request(
            "POST",
            "/v1/chunked-outputs",
            detector=detector,
            payload={
                "sink": {
                    "ref": f"i22_{detector.lower()}_chunked_result",
                    "type": "hdf_chunked",
                    "location": str(chunk_server_output_path),
                    "kwargs": {
                        "compression": CHUNK_SERVER_COMPRESSION,
                        "compression_opts": CHUNK_SERVER_COMPRESSION_LEVEL,
                    },
                },
                "subpath": run_subpath,
                "collision": "replace",
                "plan": server_plan.to_dict(),
            },
        )
        output_id = initialized_output["output_id"]
        print(f"{detector}: initialized {server_plan.total_chunks} chunks as {output_id}")

        # Every partial run replaces only the sample branch and publishes one result placement.
        chunk_runs = []
        for measurement_index, (master_path, source_path) in enumerate(chunk_server_measurements):
            for chunk_index, start in enumerate(
                range(0, CHUNK_SERVER_FRAMES_PER_MEASUREMENT, CHUNK_SERVER_CHUNK_SIZE)
            ):
                stop = min(start + CHUNK_SERVER_CHUNK_SIZE, CHUNK_SERVER_FRAMES_PER_MEASUREMENT)
                upload_i22_sample_chunk(detector, session_id, source_path, start, stop)
                chunk_spec = server_chunk_spec(
                    server_plan, measurement_index, chunk_index, start, stop
                )
                run_name = (
                    f"{master_path.stem}_{detector.lower()}_"
                    f"frames_{start:03d}_{stop:03d}"
                )
                result = api_request(
                    "POST",
                    f"/v1/sessions/{session_id}/process",
                    detector=detector,
                    payload={
                        "mode": "partial",
                        "changed_sources": ["sample"],
                        "run_name": run_name,
                        "rollback_snapshot": False,
                        "chunk_output": {
                            "output_id": output_id,
                            "chunk_spec": chunk_spec.to_dict(),
                        },
                    },
                )
                chunk_runs.append(
                    {
                        "measurement": master_path.name,
                        "frames": [start, stop],
                        "chunk_id": chunk_spec.chunk_id,
                        "run_id": result.get("run_id"),
                    }
                )
                completed = len(chunk_runs)
                if completed == 1 or completed % 5 == 0 or completed == server_plan.total_chunks:
                    progress = api_request(
                        "GET",
                        f"/v1/chunked-outputs/{output_id}",
                        detector=detector,
                    )
                    print(
                        f"{detector} {completed:02d}/{server_plan.total_chunks}: "
                        f"{chunk_spec.chunk_id}, manifest={progress['completed_chunks']}"
                    )

        # Finalization validates complete coverage and preserves all per-chunk trace groups.
        pre_finalize_status = api_request(
            "GET",
            f"/v1/chunked-outputs/{output_id}",
            detector=detector,
        )
        finalized_output = api_request(
            "POST",
            f"/v1/chunked-outputs/{output_id}/finalize",
            detector=detector,
            payload={"plan_hash": server_plan.plan_hash},
        )

        # Compare the first assembled result with the pilot and verify trace coverage.
        with h5py.File(pilot_output_path, "r") as pilot_h5, h5py.File(
            chunk_server_output_path, "r"
        ) as chunked_h5:
            pilot_signal = pilot_h5[
                f"/processing/result/{pilot_run_name}/sample/signal/signal"
            ][()]
            result_path = f"/processing/result/{run_subpath}/sample/signal/signal"
            first_chunk_signal = chunked_h5[result_path][0, 0]
            np.testing.assert_allclose(
                first_chunk_signal, pilot_signal, rtol=1e-12, atol=0, equal_nan=True
            )
            stored_shape = tuple(chunked_h5[result_path].shape)
            trace_path = f"/processing/tracer/{run_subpath}/chunks"
            trace_chunk_ids = tuple(sorted(chunked_h5[trace_path].keys()))
            if len(trace_chunk_ids) != server_plan.total_chunks:
                raise AssertionError(
                    f"{detector} stored {len(trace_chunk_ids)} trace chunks; "
                    f"expected {server_plan.total_chunks}."
                )

        return {
            "detector": detector,
            "run_subpath": run_subpath,
            "status_before_finalize": pre_finalize_status["status"],
            "status_after_finalize": finalized_output["status"],
            "completed_chunks": finalized_output["completed_chunks"],
            "expected_chunks": finalized_output["expected_chunks"],
            "trace_chunks": len(trace_chunk_ids),
            "stored_signal_shape": list(stored_shape),
            "pilot_first_chunk_comparison": "allclose",
            "output": str(chunk_server_output_path),
            "plan_id": server_plan.plan_id,
            "plan_hash": server_plan.plan_hash,
        }
    finally:
        if CHUNK_SERVER_DELETE_SESSION_AFTER_RUN:
            delete_session_if_exists(detector, session_id)
        if CHUNK_SERVER_STOP_SERVER_AFTER_RUN:
            stop_server(detector)


# SAXS and WAXS append distinct finalized run groups to the same physical file.
server_chunk_summaries = [
    process_chunked_detector(detector) for detector in CHUNK_SERVER_DETECTORS
]
display(JSON(server_chunk_summaries))


## Example 3 — Full-Scale Server-Side HDF5 Slicing

This opt-in validation runs the same four 100-frame measurements and both detectors without uploading detector arrays through HTTP. The runtime registers each preprocessed measurement as an `HDFSource` and applies the plan's typed source bindings at read time, so the detector and its three frame-aligned normalization arrays are read in ten-frame slices. Background, calibration, and mask inputs remain complete reusable HDF5 sources.

The explicit complete-plan path is intentional here. The I22 pipeline averages both input batch axes inside every chunk, while the constrained `awaiting_schema` workflow currently requires those batch axes to survive in the processed output. Example 2's real pilot file supplies the reduced output schema; Example 3 then performs every production chunk through direct server-side reads. The resulting HDFSource and BufferSource run groups are compared dataset by dataset when both examples have run.

Set `RUN_FULL_HDF_SOURCE_EXAMPLE = True` to execute the full 80-run validation. Use a smaller measurement limit or detector tuple for a quick interactive smoke check.


In [ ]:
# Keep full-scale transport tests opt-in: each enabled example performs 80 pipeline runs.
RUN_FULL_HDF_SOURCE_EXAMPLE = False
DIRECT_SOURCE_DETECTORS = tuple(CHUNK_SERVER_DETECTORS)
DIRECT_SOURCE_COMPARISON_RTOL = 1e-12


def direct_sample_registration(source_mode, source_path, *, tiled_url=None):
    """Build the changing sample registration for a direct HDF5 or Tiled read."""
    source_path = Path(source_path)
    if source_mode == "hdf":
        return {"ref": "sample", "type": "hdf", "location": str(source_path)}
    if source_mode == "tiled":
        if not tiled_url:
            raise ValueError("tiled_url is required for a Tiled sample registration.")
        return {
            "ref": "sample",
            "type": "tiled",
            "location": tiled_url,
            "kwargs": {"base_item_path": f"samples/{source_path.stem}"},
        }
    raise ValueError("source_mode must be 'hdf' or 'tiled'.")


def compare_chunked_run_groups(left_path, left_run, right_path, right_run):
    """Compare all stored BaseData arrays without loading the assembled result at once."""
    left_root = f"/processing/result/{left_run}"
    right_root = f"/processing/result/{right_run}"
    with h5py.File(left_path, "r") as left_h5, h5py.File(right_path, "r") as right_h5:
        left_group = left_h5[left_root]
        right_group = right_h5[right_root]
        left_datasets = {}
        right_datasets = {}
        left_group.visititems(
            lambda name, item: left_datasets.setdefault(name, item)
            if isinstance(item, h5py.Dataset)
            else None
        )
        right_group.visititems(
            lambda name, item: right_datasets.setdefault(name, item)
            if isinstance(item, h5py.Dataset)
            else None
        )
        if set(left_datasets) != set(right_datasets):
            raise AssertionError(
                f"Stored dataset paths differ: {sorted(left_datasets)} versus {sorted(right_datasets)}"
            )

        compared_slices = 0
        for relative_path in sorted(left_datasets):
            left = left_datasets[relative_path]
            right = right_datasets[relative_path]
            if left.shape != right.shape or left.dtype != right.dtype:
                raise AssertionError(
                    f"{relative_path} schema differs: {left.shape}/{left.dtype} versus "
                    f"{right.shape}/{right.dtype}"
                )
            # Chunk-dependent arrays start with measurement and frame-chunk dimensions.
            if left.ndim >= 2 and tuple(left.shape[:2]) == (
                len(chunk_server_measurements),
                int(np.ceil(CHUNK_SERVER_FRAMES_PER_MEASUREMENT / CHUNK_SERVER_CHUNK_SIZE)),
            ):
                selections = (
                    (measurement_index, chunk_index, ...)
                    for measurement_index in range(left.shape[0])
                    for chunk_index in range(left.shape[1])
                )
            else:
                selections = ((),)
            for selection in selections:
                left_values = left[selection]
                right_values = right[selection]
                if np.issubdtype(left.dtype, np.number):
                    np.testing.assert_allclose(
                        left_values,
                        right_values,
                        rtol=DIRECT_SOURCE_COMPARISON_RTOL,
                        atol=0,
                        equal_nan=True,
                    )
                else:
                    np.testing.assert_array_equal(left_values, right_values)
                compared_slices += 1
    return {"datasets": len(left_datasets), "compared_slices": compared_slices, "status": "allclose"}


def process_direct_source_detector(detector, source_mode, output_path, *, tiled_url=None):
    """Run one detector lifecycle while the runtime owns all changing-source slices."""
    start_server(detector)
    session_id = f"i22-{detector.lower()}-{source_mode}-frame-chunks"
    run_subpath = f"processed_{detector.lower()}_{source_mode}_frame_chunks"
    delete_session_if_exists(detector, session_id)

    first_master, first_source = chunk_server_measurements[0]
    pilot_run_name = f"{first_master.stem}_{detector.lower()}_pilot"
    pilot_output_path = chunk_server_dir / f"{pilot_run_name}.h5"
    if not pilot_output_path.is_file():
        raise FileNotFoundError(
            f"Schema pilot missing: {pilot_output_path}. Run Example 2 for {detector} first."
        )

    try:
        # A full first run seeds the reusable background/static branches; later runs only replace sample.
        api_request(
            "POST",
            "/v1/sessions",
            detector=detector,
            payload={
                "session_id": session_id,
                "name": f"I22 {detector} direct {source_mode} chunk validation",
                "pipeline": {"yaml_text": chunk_validation_pipeline_yaml(detector)},
                "trace": {"enabled": CHUNK_SERVER_TRACE_ENABLED, "watch": TRACE_WATCH},
                "auto_full_reset_on_partial_error": True,
            },
        )
        source_registrations = [
            source
            for source in build_source_registrations(detector, first_source)
            if source["ref"] != "sample"
        ]
        source_registrations.append(
            direct_sample_registration(source_mode, first_source, tiled_url=tiled_url)
        )
        api_request(
            "PUT",
            f"/v1/sessions/{session_id}/sources",
            detector=detector,
            payload={"sources": source_registrations},
        )

        server_plan = build_server_chunk_plan(
            pilot_output_path,
            pilot_run_name,
            detector,
            [master for master, _source in chunk_server_measurements],
            chunk_server_source_shapes[detector],
            source_mode=source_mode,
        )
        initialized = api_request(
            "POST",
            "/v1/chunked-outputs",
            detector=detector,
            payload={
                "sink": {
                    "ref": f"i22_{detector.lower()}_{source_mode}_chunked_result",
                    "type": "hdf_chunked",
                    "location": str(output_path),
                    "kwargs": {
                        "compression": CHUNK_SERVER_COMPRESSION,
                        "compression_opts": CHUNK_SERVER_COMPRESSION_LEVEL,
                    },
                },
                "subpath": run_subpath,
                "collision": "replace",
                "plan": server_plan.to_dict(),
            },
        )
        output_id = initialized["output_id"]
        completed = 0

        for measurement_index, (master_path, source_path) in enumerate(chunk_server_measurements):
            # Re-registering the same logical source invalidates the previous file/catalog-node cache.
            api_request(
                "PUT",
                f"/v1/sessions/{session_id}/sources",
                detector=detector,
                payload={
                    "sources": [
                        direct_sample_registration(
                            source_mode, source_path, tiled_url=tiled_url
                        )
                    ]
                },
            )
            for chunk_index, start in enumerate(
                range(0, CHUNK_SERVER_FRAMES_PER_MEASUREMENT, CHUNK_SERVER_CHUNK_SIZE)
            ):
                stop = min(start + CHUNK_SERVER_CHUNK_SIZE, CHUNK_SERVER_FRAMES_PER_MEASUREMENT)
                chunk_spec = server_chunk_spec(
                    server_plan, measurement_index, chunk_index, start, stop
                )
                process_payload = {
                    "mode": "full" if completed == 0 else "partial",
                    "run_name": (
                        f"{master_path.stem}_{detector.lower()}_{source_mode}_"
                        f"frames_{start:03d}_{stop:03d}"
                    ),
                    "rollback_snapshot": False,
                    "chunk_output": {
                        "output_id": output_id,
                        "chunk_spec": chunk_spec.to_dict(),
                    },
                }
                if completed:
                    process_payload["changed_sources"] = ["sample"]
                result = api_request(
                    "POST",
                    f"/v1/sessions/{session_id}/process",
                    detector=detector,
                    payload=process_payload,
                )
                completed += 1
                if completed == 1 or completed % 5 == 0 or completed == server_plan.total_chunks:
                    progress = api_request(
                        "GET", f"/v1/chunked-outputs/{output_id}", detector=detector
                    )
                    print(
                        f"{detector} {source_mode} {completed:02d}/{server_plan.total_chunks}: "
                        f"{chunk_spec.chunk_id}, manifest={progress['completed_chunks']}"
                    )
                if not result.get("chunk_output"):
                    raise AssertionError("The process response did not acknowledge chunk publication.")

        finalized = api_request(
            "POST",
            f"/v1/chunked-outputs/{output_id}/finalize",
            detector=detector,
            payload={"plan_hash": server_plan.plan_hash},
        )
        with h5py.File(pilot_output_path, "r") as pilot_h5, h5py.File(output_path, "r") as result_h5:
            pilot_signal = pilot_h5[
                f"/processing/result/{pilot_run_name}/sample/signal/signal"
            ][()]
            result_path = f"/processing/result/{run_subpath}/sample/signal/signal"
            np.testing.assert_allclose(
                result_h5[result_path][0, 0],
                pilot_signal,
                rtol=DIRECT_SOURCE_COMPARISON_RTOL,
                atol=0,
                equal_nan=True,
            )
            stored_shape = tuple(result_h5[result_path].shape)
            trace_chunks = len(
                result_h5[f"/processing/tracer/{run_subpath}/chunks"]
            )
            first_manifest = result_h5[
                f"/processing/chunk_plans/{server_plan.plan_id}/chunks/"
                f"{server_plan.expected_chunk_ids[0]}"
            ]
            execution = json.loads(_hdf_text(first_manifest["execution_json"][()]))
            bound_data_keys = {item["data_key"] for item in execution["source_slices"]}
            if bound_data_keys != set(sample_aligned_data_paths(detector)):
                raise AssertionError(f"Unexpected persisted source bindings: {sorted(bound_data_keys)}")

        return {
            "detector": detector,
            "source_mode": source_mode,
            "status": finalized["status"],
            "completed_chunks": finalized["completed_chunks"],
            "trace_chunks": trace_chunks,
            "bound_source_slices": len(bound_data_keys),
            "stored_signal_shape": list(stored_shape),
            "pilot_first_chunk_comparison": "allclose",
            "run_subpath": run_subpath,
            "output": str(output_path),
            "plan_id": server_plan.plan_id,
            "plan_hash": server_plan.plan_hash,
        }
    finally:
        if CHUNK_SERVER_DELETE_SESSION_AFTER_RUN:
            delete_session_if_exists(detector, session_id)
        if CHUNK_SERVER_STOP_SERVER_AFTER_RUN:
            stop_server(detector)


hdf_source_output_path = chunk_server_dir / (
    f"i22_saxs_waxs_{len(chunk_server_measurements)}x"
    f"{CHUNK_SERVER_FRAMES_PER_MEASUREMENT}_hdfsource_chunks.h5"
)
if RUN_FULL_HDF_SOURCE_EXAMPLE:
    hdf_source_summaries = [
        process_direct_source_detector(detector, "hdf", hdf_source_output_path)
        for detector in DIRECT_SOURCE_DETECTORS
    ]
    hdf_buffer_comparisons = [
        {
            "detector": summary["detector"],
            **compare_chunked_run_groups(
                chunk_server_output_path,
                f"processed_{summary['detector'].lower()}_frame_chunks",
                hdf_source_output_path,
                summary["run_subpath"],
            ),
        }
        for summary in hdf_source_summaries
    ]
    display(JSON({"runs": hdf_source_summaries, "versus_buffer": hdf_buffer_comparisons}))
else:
    print("Set RUN_FULL_HDF_SOURCE_EXAMPLE = True to run Example 3.")


## Example 4 — The Same Full-Scale Run Through A Local Tiled Server

This opt-in example exposes only the changing sample arrays from the packaged preprocessed files through a notebook-owned, read-only Tiled server. MoDaCor registers a `TiledSource` for each measurement and applies the same typed slice bindings used by Example 3. The complete background, calibration, and masks remain direct HDF5 sources so this comparison isolates the changing-source transport.

The local catalogue uses Tiled's HDF5 adapters, so it does not copy the detector files into another store. After both examples complete, every stored result dataset is compared chunk by chunk against the HDFSource output. This exercises real HTTP serialization, Tiled metadata/structure discovery, backend slicing, MoDaCor partial reruns, HDF5 chunk publication, traces, and finalization.

This requires the `tiled-tests` MoDaCor extra shown in the setup command. Set `RUN_TILED_SOURCE_EXAMPLE = True` to execute it.


In [ ]:
# The Tiled service is kept separate from the MoDaCor runtime service.
RUN_TILED_SOURCE_EXAMPLE = False
TILED_SERVER_HOST = "127.0.0.1"
TILED_SERVER_PORT = 8910
TILED_SERVER_URL = f"http://{TILED_SERVER_HOST}:{TILED_SERVER_PORT}"
# Preserve a running notebook-owned service when this cell is executed again.
TILED_SERVER = globals().get("TILED_SERVER")
TILED_SERVER_THREAD = globals().get("TILED_SERVER_THREAD")


def _nested_tiled_mapping(mapping):
    from tiled.adapters.mapping import MapAdapter

    return MapAdapter(
        {
            key: _nested_tiled_mapping(value) if isinstance(value, dict) else value
            for key, value in mapping.items()
        }
    )


def _tiled_measurement_adapter(source_path):
    """Expose only sample arrays used by the two I22 pipelines."""
    from tiled.adapters.hdf5 import HDF5Adapter

    source_path = Path(source_path).resolve()
    data_paths = {"/modacor/calibration/absolute_intensity_factor"}
    for detector in CHUNK_SERVER_DETECTORS:
        data_paths.update(sample_aligned_data_paths(detector))

    tree = {}
    for data_path in sorted(data_paths):
        cursor = tree
        parts = data_path.strip("/").split("/")
        for part in parts[:-1]:
            cursor = cursor.setdefault(part, {})
        cursor[parts[-1]] = HDF5Adapter.from_uris(
            source_path.as_uri(), dataset=data_path
        )
    return _nested_tiled_mapping(tree)


def build_i22_tiled_tree():
    """Build a read-only catalogue whose leaves stream from the packaged HDF5 files."""
    from tiled.adapters.mapping import MapAdapter

    samples = {
        source_path.stem: _tiled_measurement_adapter(source_path)
        for _master_path, source_path in chunk_server_measurements
    }
    return MapAdapter({"samples": MapAdapter(samples)})


def start_local_tiled_server(timeout_s=45.0):
    global TILED_SERVER, TILED_SERVER_THREAD
    if (
        TILED_SERVER is not None
        and TILED_SERVER_THREAD is not None
        and TILED_SERVER_THREAD.is_alive()
    ):
        return TILED_SERVER

    try:
        import uvicorn
        from tiled.client import from_uri
        from tiled.config import Authentication
        from tiled.server.app import build_app
    except ImportError as exc:
        raise RuntimeError(
            "Example 4 requires the MoDaCor tiled-tests extra; install it and restart the kernel."
        ) from exc
    from threading import Thread

    # This loopback-only demonstrator is deliberately anonymous; facility deployments should authenticate.
    app = build_app(
        build_i22_tiled_tree(),
        authentication=Authentication(allow_anonymous_access=True),
    )
    config = uvicorn.Config(
        app,
        host=TILED_SERVER_HOST,
        port=TILED_SERVER_PORT,
        log_level="warning",
        access_log=False,
    )
    TILED_SERVER = uvicorn.Server(config)
    TILED_SERVER_THREAD = Thread(
        target=TILED_SERVER.run,
        name="i22-notebook-tiled",
        daemon=True,
    )
    TILED_SERVER_THREAD.start()

    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        if TILED_SERVER.started:
            client = from_uri(TILED_SERVER_URL)
            print(
                f"Tiled server ready at {TILED_SERVER_URL}; "
                f"measurements={len(list(client['samples']))}"
            )
            return TILED_SERVER
        if not TILED_SERVER_THREAD.is_alive():
            raise RuntimeError("The notebook-owned Tiled server exited during startup.")
        time.sleep(0.25)
    raise TimeoutError(f"Tiled did not start within {timeout_s:.0f} seconds.")


def stop_local_tiled_server():
    global TILED_SERVER, TILED_SERVER_THREAD
    if TILED_SERVER is None:
        return
    TILED_SERVER.should_exit = True
    if TILED_SERVER_THREAD is not None:
        TILED_SERVER_THREAD.join(timeout=10)
    TILED_SERVER = None
    TILED_SERVER_THREAD = None
    print("Stopped notebook-owned Tiled server.")


atexit.register(stop_local_tiled_server)

tiled_source_output_path = chunk_server_dir / (
    f"i22_saxs_waxs_{len(chunk_server_measurements)}x"
    f"{CHUNK_SERVER_FRAMES_PER_MEASUREMENT}_tiledsource_chunks.h5"
)
if RUN_TILED_SOURCE_EXAMPLE:
    if not RUN_FULL_HDF_SOURCE_EXAMPLE and not hdf_source_output_path.is_file():
        raise FileNotFoundError(
            "Run Example 3 first so Example 4 has a direct-HDF reference result."
        )
    start_local_tiled_server()
    try:
        tiled_source_summaries = [
            process_direct_source_detector(
                detector,
                "tiled",
                tiled_source_output_path,
                tiled_url=TILED_SERVER_URL,
            )
            for detector in DIRECT_SOURCE_DETECTORS
        ]
        tiled_hdf_comparisons = [
            {
                "detector": summary["detector"],
                **compare_chunked_run_groups(
                    hdf_source_output_path,
                    f"processed_{summary['detector'].lower()}_hdf_frame_chunks",
                    tiled_source_output_path,
                    summary["run_subpath"],
                ),
            }
            for summary in tiled_source_summaries
        ]
        display(JSON({"runs": tiled_source_summaries, "versus_hdf": tiled_hdf_comparisons}))
    finally:
        stop_local_tiled_server()
else:
    print("Set RUN_TILED_SOURCE_EXAMPLE = True to run Example 4.")


## Plot A Processed Output File

This final cell loads one server output HDF5 file and plots the final sample signal against Q. It uses x error bars from Q uncertainty when available, and y error bars from signal uncertainty when available.


In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np


def _decode_hdf_attr(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    if isinstance(value, np.ndarray) and value.shape == ():
        return _decode_hdf_attr(value.item())
    return value


def _latest_output_file():
    if "run_results" in globals():
        successful_outputs = [Path(item["output"]) for item in run_results if Path(item["output"]).exists()]
        if successful_outputs:
            return successful_outputs[-1]

    output_files = sorted(OUTPUT_DIR.glob("*_server_result.h5"), key=lambda item: item.stat().st_mtime)
    if not output_files:
        raise FileNotFoundError(f"No server output files found in {OUTPUT_DIR}")
    return output_files[-1]


def _default_child(group):
    default_name = _decode_hdf_attr(group.attrs.get("default"))
    if default_name in group:
        return group[default_name], str(default_name)
    first_name = next(iter(group.keys()))
    return group[first_name], str(first_name)


def _uncertainty(group, preferred_names):
    uncertainties = group.get("uncertainties")
    if uncertainties is None:
        return None, None
    for name in preferred_names:
        if name in uncertainties:
            return uncertainties[name][()], name
    return None, None


plot_output_path = _latest_output_file()

with h5py.File(plot_output_path, "r") as h5:
    result_group, run_name = _default_child(h5["processing/result"])
    sample_group = result_group["sample"]
    signal_group = sample_group["signal"]

    q = signal_group["Q"][()] if "Q" in signal_group else sample_group["Q"]["signal"][()]
    signal = signal_group["signal"][()]
    q_units = _decode_hdf_attr(signal_group.get("Q", sample_group["Q"]["signal"]).attrs.get("units", ""))
    signal_units = _decode_hdf_attr(signal_group["signal"].attrs.get("units", ""))

    xerr = None
    xerr_name = None
    if "Q" in sample_group:
        xerr, xerr_name = _uncertainty(sample_group["Q"], ["uncertainty_combined", "SEM", "STD"])

    yerr, yerr_name = _uncertainty(signal_group, ["uncertainty_pixelvalues", "SEM", "STD", "poisson"])

valid = np.isfinite(q) & np.isfinite(signal)
if xerr is not None:
    valid &= np.isfinite(xerr)
if yerr is not None:
    valid &= np.isfinite(yerr)

fig, ax = plt.subplots(figsize=(7.5, 4.8), constrained_layout=True)
ax.errorbar(
    q[valid],
    signal[valid],
    xerr=xerr[valid] if xerr is not None else None,
    yerr=yerr[valid] if yerr is not None else None,
    fmt="o",
    linestyle="none",
    markersize=3,
    elinewidth=0.7,
    capsize=1.5,
    alpha=0.8,
)
if np.all(q[valid] > 0):
    ax.set_xscale("log")
if np.all(signal[valid] > 0):
    ax.set_yscale("log")
ax.set_xlabel(f"Q ({q_units})" if q_units else "Q")
ax.set_ylabel(f"Signal ({signal_units})" if signal_units else "Signal")
ax.set_title(plot_output_path.name)
ax.grid(True, which="both", alpha=0.25)

print(f"Loaded: {plot_output_path}")
print(f"Run: {run_name}")
print(f"x error: {xerr_name or 'not available'}")
print(f"y error: {yerr_name or 'not available'}")
plt.show()
